# Cùng một RAG, hai cách viết: SDK trực tiếp và LangChain

Bài Intermediate 02 viết pipeline RAG bằng tay: retrieve hai tầng, prompt có nguồn đánh số, cổng từ chối, đọc trích
dẫn. Ở đây bạn dựng lại **đúng pipeline đó** bằng LangChain (LCEL, `langchain-ollama`), rồi so hai bản trên 50 câu ghim:
số dòng code, độ trễ, chất lượng, và trải nghiệm debug. Kết quả là `COMPARE.md`: một khuyến nghị có số liệu, không
phải cảm nhận.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Test gọi `build_messages`, `parse_citations`, `answer` (bản SDK) và
  `build_chain` (bản LangChain) với LLM giả: `FakeLLM` của `llmkit` và `FakeListChatModel` của LangChain.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b` và `bge-m3`, và cache vector corpus của bài Intermediate 01
  (`data/raw/zalo-legal/cache/bge-m3.npy`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/13-rag/exercises/advanced/02-rag-frameworks -q`

In [ ]:
# explore: tìm gốc repo, tải Zalo Legal nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"
print(ZALO, sorted(p.name for p in CACHE.iterdir()) if CACHE.exists() else "chưa có cache: làm bài Intermediate 01 trước")

In [ ]:
import json
import re
import time
from dataclasses import dataclass, field
from typing import Callable

import numpy as np
from langchain_core.documents import Document
from langchain_core.language_models import BaseChatModel
from langchain_core.runnables import Runnable

import llmkit
from llmkit import LLMClient, Message

## 1. Bản SDK trực tiếp (dán từ bài Intermediate 02)

**Dán** `build_messages`, `parse_citations`, `answer` của bạn ở bài Intermediate 02 vào cell dưới, giữ nguyên chữ ký.
Đây là mốc để so: bản LangChain phải cho **cùng prompt và cùng kết quả**. Test kiểm lại ba hàm này bằng `FakeLLM`.

In [ ]:
REFUSAL = "Không tìm thấy căn cứ trong tài liệu được cung cấp."


@dataclass
class Article:
    uid: str      # "<số hiệu văn bản>+<số điều>"
    title: str
    text: str     # nội dung đưa vào prompt (một khoản)


@dataclass
class RagAnswer:
    text: str
    citations: list[str] = field(default_factory=list)   # uid các điều được trích, theo thứ tự, không trùng
    refused: bool = False


def build_messages(question: str, passages: list[Article]) -> list[Message]:
    """[system, user] như bài Intermediate 02."""
    # TODO: dán lời giải bài Intermediate 02
    raise NotImplementedError


def parse_citations(answer: str, n_passages: int) -> list[int]:
    """Các số [i] theo thứ tự xuất hiện, chỉ giữ 1..n_passages, bỏ trùng (Intermediate 02)."""
    # TODO: dán lời giải bài Intermediate 02
    raise NotImplementedError


def answer(question: str, retrieve: Callable[[str], list[tuple[Article, float]]],
           llm: LLMClient, min_score: float) -> RagAnswer:
    """Cổng điểm trước LLM, gọi LLM một lần, map [i] về uid (Intermediate 02)."""
    # TODO: dán lời giải bài Intermediate 02
    raise NotImplementedError

## 2. Bản LangChain

Trong LangChain, retriever trả `Document`. Quy ước của bài: mỗi nguồn là một `Document` với `page_content` là nội dung
khoản, `metadata` có `uid`, `title`, `score` (cosine). `to_documents` đổi kết quả retrieve kiểu SDK sang dạng này.

`build_chain(retriever, llm, min_score=0.5) -> Runnable`:

- `retriever`: bất kỳ `Runnable` nhận câu hỏi (`str`) trả `list[Document]`: một `BaseRetriever` khi chạy thật,
  `RunnableLambda` trong test. Gọi **đúng một lần** mỗi câu hỏi.
- `llm`: một chat model của LangChain: `ChatOllama` khi chạy thật, `FakeListChatModel` trong test.
- `chain.invoke(question)` trả `RagAnswer` **giống hệt** `answer(question, ...)` của bản SDK khi model trả cùng văn bản:
  - **cổng** trước LLM: không có `Document` nào hoặc mọi `score` dưới `min_score` thì trả `RagAnswer(REFUSAL, [], True)`
    và **không gọi** model;
  - model nhận đúng **hai** message, `system` và `human`, nội dung bằng hệt `build_messages(question, passages)`
    (cùng prompt thì so độ trễ, chất lượng mới công bằng);
  - trích dẫn map về `uid` bằng `parse_citations`, bỏ uid trùng; `refused` khi câu trả lời chứa "Không tìm thấy căn cứ".
- Dựng bằng LCEL: ghép các bước bằng `|`, `RunnablePassthrough.assign`, `RunnableBranch` hoặc `RunnableLambda`,
  `StrOutputParser`. **Không** gọi hàm `answer` của bản SDK bên trong chain: bài này so hai cách viết, gọi lại bản SDK
  thì không còn gì để so. Được dùng lại `build_messages` và `parse_citations` (prompt và cách đọc trích dẫn là của bạn,
  không phải của framework).

In [ ]:
def to_documents(hits: list[tuple[Article, float]]) -> list[Document]:
    """Kết quả retrieve kiểu SDK -> Document của LangChain (metadata: uid, title, score)."""
    return [Document(page_content=a.text, metadata={"uid": a.uid, "title": a.title, "score": s}) for a, s in hits]


def build_chain(retriever: Runnable, llm: BaseChatModel, min_score: float = 0.5) -> Runnable:
    """LCEL: retriever -> cổng điểm -> prompt (bằng hệt build_messages) -> llm -> RagAnswer."""
    # TODO
    raise NotImplementedError

## 3. So hai bản trên dữ liệu thật

Các cell dưới:

1. Dựng lại retrieve hai tầng của bài Intermediate 02 (dense `bge-m3` tìm 5 điều, giữ 5 khoản gần câu hỏi nhất) theo
   kiểu SDK, và **một retriever LangChain** làm cùng việc (`BaseRetriever` + `OllamaEmbeddings`). Kiểm hai bên trả
   cùng nguồn.
2. Chạy 50 câu đầu tập ghim qua **cả hai bản với cùng nguồn** (retrieve một lần, đưa cùng `hits` cho hai bên), xen kẽ
   thứ tự gọi để không bên nào được lợi vì model đã nạp sẵn. Đo độ trễ phần sinh, số câu trả lời giống hệt nhau, và các
   thước đo chất lượng của bài Intermediate 02. Khoảng 100 lượt gọi `qwen2.5:3b`, 10 phút trên laptop GPU 4GB.
3. Đo **chi phí riêng của framework** bằng LLM giả (không có thời gian model), đếm số dòng code, và xem LangChain gửi gì
   cho model.

In [ ]:
# explore: corpus, 50 câu đầu tập ghim, cache vector corpus; retrieve hai tầng kiểu SDK (như bài Intermediate 02)
N_EVAL = 50
corpus: dict[str, Article] = {}
with open(ZALO / "corpus.jsonl", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        corpus[d["_id"]] = Article(d["_id"], d["title"], d["text"])
qrels: dict[str, set[str]] = {}
with open(ZALO / "qrels" / "test.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        qrels.setdefault(row["query-id"], set()).add(row["corpus-id"])
with open(ZALO / "queries.jsonl", encoding="utf-8") as f:
    query_text = {q["_id"]: q["text"] for q in map(json.loads, f)}
pinned = list(qrels)[:N_EVAL]

if not (CACHE / "bge-m3.npy").exists():
    raise RuntimeError("Chưa có cache vector corpus: chạy notebook bài Intermediate 01 trước.")
uids = json.loads((CACHE / "bge-m3.ids.json").read_text(encoding="utf-8"))
matrix = np.load(CACHE / "bge-m3.npy")
KHOAN = re.compile(r"\d+\.\s")


def split_khoan(a: Article) -> list[Article]:
    groups: list[list[str]] = []
    for line in a.text.split("\n"):
        if not line.strip():
            continue
        if KHOAN.match(line.strip()) or not groups:
            groups.append([line.strip()])
        else:
            groups[-1].append(line.strip())
    return [Article(a.uid, a.title, "\n".join(g)[:2000]) for g in groups] or [a]


def unit(v: np.ndarray) -> np.ndarray:
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def retrieve(question: str, n_articles: int = 5, n_passages: int = 5) -> list[tuple[Article, float]]:
    q = unit(llmkit.embed([question])[0])
    top = np.argsort(-(matrix @ q), kind="stable")[:n_articles]
    chunks = [c for i in top for c in split_khoan(corpus[uids[i]])]
    scores = unit(llmkit.embed([c.title + "\n" + c.text for c in chunks])) @ q
    best = np.argsort(-scores, kind="stable")[:n_passages]
    return [(chunks[i], float(scores[i])) for i in best]


t0 = time.perf_counter()
HITS = {qid: retrieve(query_text[qid]) for qid in pinned}
print(f"retrieve SDK: {(time.perf_counter() - t0) / len(pinned) * 1000:.0f} ms/câu")

In [ ]:
# explore: retriever LangChain làm cùng việc: BaseRetriever + OllamaEmbeddings (langchain-ollama) trên cùng ma trận
from langchain_core.retrievers import BaseRetriever
from langchain_ollama import OllamaEmbeddings


class DenseKhoanRetriever(BaseRetriever):
    """Dense 5 điều trên `matrix`, tách khoản, trả 5 Document khoản gần câu hỏi nhất (metadata uid, title, score)."""

    embeddings: OllamaEmbeddings
    n_articles: int = 5
    n_passages: int = 5

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> list[Document]:
        # TODO: như retrieve() ở trên nhưng embed bằng self.embeddings (embed_query, embed_documents)
        raise NotImplementedError


lc_retriever = DenseKhoanRetriever(embeddings=OllamaEmbeddings(model="bge-m3"))
same = 0
t0 = time.perf_counter()
for qid in pinned[:10]:
    docs = lc_retriever.invoke(query_text[qid])
    same += [d.metadata["uid"] for d in docs] == [a.uid for a, _ in HITS[qid]]
print(f"retriever LangChain: {(time.perf_counter() - t0) / 10 * 1000:.0f} ms/câu; cùng nguồn với bản SDK: {same}/10")

In [ ]:
# explore: 50 câu, cùng nguồn cho hai bản, xen kẽ thứ tự gọi; đo độ trễ phần sinh và chất lượng
from langchain_core.runnables import RunnableLambda
from langchain_ollama import ChatOllama

MODEL, MIN_SCORE = "qwen2.5:3b", 0.5
chat_model = ChatOllama(model=MODEL, temperature=0)
sdk_client = llmkit.get_client()          # mặc định qwen2.5:3b; đặt LLM_MODEL nếu đổi model
rows, TEXTS = [], {}
for n, qid in enumerate(pinned):
    q, hits = query_text[qid], HITS[qid]
    chain = build_chain(RunnableLambda(lambda _q, docs=to_documents(hits): docs), chat_model, MIN_SCORE)
    runs = {}
    for side in (["sdk", "lc"] if n % 2 == 0 else ["lc", "sdk"]):
        t0 = time.perf_counter()
        runs[side] = (answer(q, lambda _q: hits, sdk_client, MIN_SCORE) if side == "sdk" else chain.invoke(q))
        runs[side + "_s"] = time.perf_counter() - t0
    gold_in = bool(qrels[qid] & {a.uid for a, _ in hits})
    TEXTS[qid] = runs["sdk"].text
    rows.append({"qid": qid, "sdk_s": runs["sdk_s"], "lc_s": runs["lc_s"], "gold_in": gold_in,
                 "same_text": runs["sdk"].text == runs["lc"].text,
                 **{f"{s}_{k}": v for s in ("sdk", "lc") for k, v in
                    {"cited": bool(runs[s].citations), "cited_gold": bool(qrels[qid] & set(runs[s].citations)),
                     "refused": runs[s].refused}.items()}})
    if not rows[-1]["same_text"]:
        print(f"khác nhau ở {qid}:\n  SDK: {runs['sdk'].text[:150]!r}\n  LC:  {runs['lc'].text[:150]!r}")

import pandas as pd

df = pd.DataFrame(rows)
print(f"độ trễ trung vị: SDK {df.sdk_s.median():.2f} s, LangChain {df.lc_s.median():.2f} s | "
      f"câu trả lời giống hệt: {df.same_text.sum()}/{len(df)} | điều đúng trong nguồn: {df.gold_in.sum()}")
display(pd.DataFrame({side: {"có trích dẫn": df[f"{side}_cited"].sum(), "trích đúng điều": df[f"{side}_cited_gold"].sum(),
                             "từ chối": df[f"{side}_refused"].sum()} for side in ("sdk", "lc")}))

In [ ]:
# explore: bản SDK tự lệch bao nhiêu? Gọi lại 10 câu đầu bằng chính bản SDK, so với lần một (cùng mọi thứ).
# Nếu chính nó cũng lệch, "khác nhau giữa hai bản" ở trên không chứng minh được framework làm đổi câu trả lời.
again = sum(answer(query_text[q], lambda _q, h=HITS[q]: h, sdk_client, MIN_SCORE).text == TEXTS[q] for q in pinned[:10])
print(f"SDK gọi lại: {again}/10 câu giống hệt lần một")

In [ ]:
# explore: chi phí riêng của framework, đo bằng LLM giả (không có thời gian model)
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from llmkit import FakeLLM

hits = HITS[pinned[0]]
fake_chain = build_chain(RunnableLambda(lambda _q: to_documents(hits)), FakeListChatModel(responses=["Theo [1]."]))
fake_sdk = FakeLLM(lambda messages: "Theo [1].")
for name, call in {"SDK": lambda: answer("câu hỏi", lambda _q: hits, fake_sdk, 0.5),
                   "LangChain": lambda: fake_chain.invoke("câu hỏi")}.items():
    call()                                   # khởi động
    t0 = time.perf_counter()
    for _ in range(300):
        call()
    print(f"{name}: {(time.perf_counter() - t0) / 300 * 1000:.3f} ms mỗi lần gọi (không tính model)")

In [ ]:
# explore: số dòng code (bỏ dòng trống, comment, docstring) của phần riêng mỗi bản
import ast
import inspect
import textwrap


def loc(*functions) -> int:
    count = 0
    for fn in functions:
        src = textwrap.dedent(inspect.getsource(fn))
        skip: set[int] = set()
        for node in ast.walk(ast.parse(src)):
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and ast.get_docstring(node):
                skip.update(range(node.body[0].lineno, node.body[0].end_lineno + 1))
        count += sum(1 for i, line in enumerate(src.splitlines(), start=1)
                     if i not in skip and line.strip() and not line.strip().startswith("#"))
    return count


print("SDK: answer =", loc(answer), "dòng | retrieve =", loc(retrieve), "dòng")
print("LangChain: build_chain + to_documents =", loc(build_chain, to_documents), "dòng | retriever =",
      loc(DenseKhoanRetriever._get_relevant_documents), "dòng (cộng khai báo lớp)")
print("dùng chung: build_messages + parse_citations =", loc(build_messages, parse_citations), "dòng")

In [ ]:
# explore: LangChain gửi gì cho model? Bật debug cho một lần gọi (in mọi bước của chain kèm input/output)
from langchain_core.globals import set_debug

set_debug(True)
_ = build_chain(RunnableLambda(lambda _q: to_documents(HITS[pinned[0]])),
                FakeListChatModel(responses=["Theo [1]."])).invoke(query_text[pinned[0]])
set_debug(False)

## Đọc kết quả và viết `COMPARE.md`

Tự trả lời trước khi viết:

- Cùng prompt, cùng model, `temperature = 0`: hai bản có cho câu trả lời giống hệt nhau không? Câu nào khác thì vì sao
  (tham số mặc định nào của `ChatOllama` khác `llmkit`)?
- Chi phí riêng của framework (mili giây) so với thời gian model (giây) chênh bao nhiêu lần? Độ trễ có phải lý do để
  chọn hay bỏ framework ở pipeline này không?
- Bản nào ít dòng hơn, và số dòng có phải thước đo đúng không: đọc lại `build_chain` sau một tuần, bạn hiểu bản nào
  nhanh hơn? Khi một câu trả lời sai, bạn lần ra prompt thật nhanh hơn ở bản nào?
- LangChain thêm bao nhiêu gói phụ thuộc (`uv pip list | grep -i -E "langchain|langsmith|ollama"`)? Khi nào thứ nó cho
  (tích hợp sẵn, streaming, callback, LangSmith, agent ở module 17) đáng giá hơn chi phí đó?

Rồi điền `COMPARE.md` (cùng thư mục) và làm phần **trắc nghiệm** của bài trong app.